In [ ]:
#!/usr/bin/env python

import numpy as np
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud
from nuscenes.utils.geometry_utils import transform_matrix # Used implicitly by Quaternion
from pyquaternion import Quaternion
import k3d # Changed from open3d
import sys
from pathlib import Path
import colorsys # For generating distinct colors if needed

# --- Configuration ---
nuscenes_dataroot_raw = '/datasets/nuscenes' # <<< ADJUST IF NEEDED
nuscenes_version = 'v1.0-mini'
scene_index = 0
current_frame_seq_id = 5 # Frame we are processing (where point 5 is defined)
historical_map_seq_id = 4 # Map/Frame we are checking against
point_original_index = 5 # Point we are debugging
output_html_file = "projection_visualization.html" # K3D output file

# --- C++ Log Values (UPDATED FROM LATEST CTEST LOGS) ---
# Find the logs for Frame 5, Point 5, Checking MapIdx=4
cpp_log_values = {
    "p_world_glob": (408.084, 1159.242, -0.086), # From "Input p_world.glob: (...)"
    "map_project_T": (15.703, 1231.094, 43.124), # From "Storing project_T (World->Sensor):" for Map 4 OR "Map Pose T:" when checking Map 4
    "p_map_frame_local": (-4.148, 3.102, -1.750)  # From "Local Coords (Rel Map): X=..., Y=..., Z=..." when checking Map 4
}
# --- End Configuration ---

# --- Helper Functions (get_sensor_to_world_matrix, load_pc - same as before) ---
def get_sensor_to_world_matrix(nusc, sample_data_token):
    """Gets the transformation matrix from sensor coordinates to world coordinates."""
    # Use 4x4 matrix method for consistency with test_nuscenes.py
    try:
        sd_record = nusc.get('sample_data', sample_data_token)
        cs_record = nusc.get('calibrated_sensor', sd_record['calibrated_sensor_token'])
        ep_record = nusc.get('ego_pose', sd_record['ego_pose_token'])

        # Sensor to Ego Transformation
        sens_to_ego_trans = np.array(cs_record['translation'])
        sens_to_ego_rot = Quaternion(cs_record['rotation'])
        sens_to_ego_matrix = np.eye(4)
        sens_to_ego_matrix[:3, :3] = sens_to_ego_rot.rotation_matrix
        sens_to_ego_matrix[:3, 3] = sens_to_ego_trans

        # Ego to Global Transformation
        ego_to_glob_trans = np.array(ep_record['translation'])
        ego_to_glob_rot = Quaternion(ep_record['rotation'])
        ego_to_glob_matrix = np.eye(4)
        ego_to_glob_matrix[:3, :3] = ego_to_glob_rot.rotation_matrix
        ego_to_glob_matrix[:3, 3] = ego_to_glob_trans

        # Sensor to Global Transformation
        sens_to_world_matrix = ego_to_glob_matrix @ sens_to_ego_matrix
        return sens_to_world_matrix
    except Exception as e:
        print(f"Error getting sensor global pose matrix for token {sample_data_token}: {e}")
        raise

def load_pc(nusc, sample_data_token):
    """Loads point cloud data for a given sample_data token."""
    lidar_filepath = Path(nusc.get_sample_data_path(sample_data_token))
    if not lidar_filepath.exists():
        print(f"Error: Lidar file not found: {lidar_filepath}")
        sys.exit(1)
    pc = LidarPointCloud.from_file(str(lidar_filepath))
    return pc.points.T[:, :3] # Return Nx3 points (X, Y, Z)

# --- Main Script Logic ---
def main():
    print("--- Starting Python Verification ---")
    print(f"NuScenes Root: {nuscenes_dataroot_raw}, Version: {nuscenes_version}")
    print(f"Scene Index: {scene_index}")
    print(f"Current Frame Seq ID: {current_frame_seq_id}")
    print(f"Historical Map Seq ID: {historical_map_seq_id}")
    print(f"Point Original Index: {point_original_index}")
    print("Using C++ Log Values:")
    for key, val in cpp_log_values.items():
        print(f"  {key}: {val}")
    print("-" * 30)

    # --- Load NuScenes ---
    try:
        nuscenes_dataroot = Path(nuscenes_dataroot_raw).expanduser()
        if not nuscenes_dataroot.exists():
            print(f"Error: Dataroot does not exist: {nuscenes_dataroot}")
            sys.exit(1)
        print(f"Loading NuScenes ({nuscenes_version} from {nuscenes_dataroot})...")
        nusc = NuScenes(version=nuscenes_version, dataroot=str(nuscenes_dataroot), verbose=False)
    except Exception as e:
        print(f"Error loading NuScenes: {e}")
        sys.exit(1)

    # --- Get Sample Tokens ---
    print("Getting sample tokens...")
    try:
        scene_info = nusc.scene[scene_index]
        sample_tokens = []
        current_token = scene_info['first_sample_token']
        max_seq_id = max(current_frame_seq_id, historical_map_seq_id)
        for i in range(max_seq_id + 1):
            if not current_token:
                print(f"Error: Scene only has {len(sample_tokens)} samples, needed {max_seq_id + 1}.")
                sys.exit(1)
            sample_tokens.append(current_token)
            sample_record = nusc.get('sample', current_token)
            current_token = sample_record['next']
        current_sample_token = sample_tokens[current_frame_seq_id]
        historical_sample_token = sample_tokens[historical_map_seq_id]
        current_sample_record = nusc.get('sample', current_sample_token)
        historical_sample_record = nusc.get('sample', historical_sample_token)
        current_lidar_token = current_sample_record['data']['LIDAR_TOP']
        historical_lidar_token = historical_sample_record['data']['LIDAR_TOP']
        print(f"Current Lidar Token (Frame {current_frame_seq_id}): {current_lidar_token}")
        print(f"Historical Lidar Token (Frame {historical_map_seq_id}): {historical_lidar_token}")
    except IndexError:
         print(f"Error: Could not retrieve sample tokens up to index {max_seq_id}. Check scene_index and frame IDs.")
         sys.exit(1)
    except Exception as e:
        print(f"Error getting sample tokens: {e}")
        sys.exit(1)
    print("-" * 30)

    # --- Get Poses ---
    print("Calculating transformations...")
    try:
        sens5_to_world_matrix = get_sensor_to_world_matrix(nusc, current_lidar_token)
        sens4_to_world_matrix = get_sensor_to_world_matrix(nusc, historical_lidar_token)
        world_to_sens5_matrix = np.linalg.inv(sens5_to_world_matrix)
        world_to_sens4_matrix = np.linalg.inv(sens4_to_world_matrix)
        map_project_R_py = world_to_sens4_matrix[:3, :3] # World -> Sensor4 Rotation
        map_project_T_py = world_to_sens4_matrix[:3, 3] # World -> Sensor4 Translation
    except Exception as e:
        print(f"Error calculating transformations: {e}")
        sys.exit(1)

    # --- Load Point Clouds ---
    print("Loading point clouds...")
    try:
        points_sens5 = load_pc(nusc, current_lidar_token)
        points_sens4 = load_pc(nusc, historical_lidar_token)
        if point_original_index >= len(points_sens5):
            print(f"Error: Point index {point_original_index} is out of bounds for Frame 5 (size {len(points_sens5)}).")
            sys.exit(1)
    except Exception as e:
        print(f"Error loading point clouds: {e}")
        sys.exit(1)

    # --- Get Specific Point & Transform ---
    print(f"Extracting point {point_original_index} from Frame 5...")
    point_sens5_xyz = points_sens5[point_original_index]
    point_sens5_hom = np.append(point_sens5_xyz, 1)
    p_world_glob_py = (sens5_to_world_matrix @ point_sens5_hom)[:3]

    # --- Verifications ---
    print("-" * 30)
    print("Verification 1: Global Point Coordinates")
    print(f"  Python calculated p_world_glob: ({p_world_glob_py[0]:.3f}, {p_world_glob_py[1]:.3f}, {p_world_glob_py[2]:.3f})")
    print(f"  C++ logged p_world.glob:      ({cpp_log_values['p_world_glob'][0]:.3f}, {cpp_log_values['p_world_glob'][1]:.3f}, {cpp_log_values['p_world_glob'][2]:.3f})")
    diff_glob = np.array(p_world_glob_py) - np.array(cpp_log_values['p_world_glob'])
    print(f"  Difference (Py - C++):        ({diff_glob[0]:.3f}, {diff_glob[1]:.3f}, {diff_glob[2]:.3f})")
    if np.allclose(p_world_glob_py, cpp_log_values['p_world_glob'], atol=1e-3): print("  >>> Global coordinates MATCH.")
    else: print("  >>> WARNING: Global coordinates DO NOT MATCH significantly!")
    print("-" * 30)

    print("Verification 2: World -> Sensor4 Translation (map.project_T)")
    print(f"  Python calculated map.project_T: ({map_project_T_py[0]:.3f}, {map_project_T_py[1]:.3f}, {map_project_T_py[2]:.3f})")
    print(f"  C++ logged map.project_T:      ({cpp_log_values['map_project_T'][0]:.3f}, {cpp_log_values['map_project_T'][1]:.3f}, {cpp_log_values['map_project_T'][2]:.3f})")
    diff_T = np.array(map_project_T_py) - np.array(cpp_log_values['map_project_T'])
    print(f"  Difference (Py - C++):         ({diff_T[0]:.3f}, {diff_T[1]:.3f}, {diff_T[2]:.3f})")
    # Increase tolerance slightly for potential float precision differences in inverse calculation
    if np.allclose(map_project_T_py, cpp_log_values['map_project_T'], atol=1e-2): print("  >>> World->Sensor4 Translation MATCHES.")
    else: print("  >>> WARNING: World->Sensor4 Translation DOES NOT MATCH significantly!")
    print("-" * 30)

    print("Verification 3: Projected Point Coordinates (in Sensor4 Frame)")
    p_world_glob_py_hom = np.append(p_world_glob_py, 1)
    p_map_frame_local_py_hom = world_to_sens4_matrix @ p_world_glob_py_hom
    p_map_frame_local_py = p_map_frame_local_py_hom[:3]
    print(f"  Python calculated p_map_frame_local: ({p_map_frame_local_py[0]:.3f}, {p_map_frame_local_py[1]:.3f}, {p_map_frame_local_py[2]:.3f})")
    print(f"  C++ logged Local Coords (Rel Map): ({cpp_log_values['p_map_frame_local'][0]:.3f}, {cpp_log_values['p_map_frame_local'][1]:.3f}, {cpp_log_values['p_map_frame_local'][2]:.3f})")
    diff_local = np.array(p_map_frame_local_py) - np.array(cpp_log_values['p_map_frame_local'])
    print(f"  Difference (Py - C++):             ({diff_local[0]:.3f}, {diff_local[1]:.3f}, {diff_local[2]:.3f})")
    if np.allclose(p_map_frame_local_py, cpp_log_values['p_map_frame_local'], atol=1e-2): print("  >>> Projected coordinates MATCH.")
    else:
        print("  >>> ERROR: Projected coordinates DO NOT MATCH significantly!")
        # Add Z sign check as before
        if p_map_frame_local_py[2] * cpp_log_values['p_map_frame_local'][2] < 0:
             print(f"  >>> NOTE: Z coordinates have different signs! Py: {p_map_frame_local_py[2]:.3f}, C++: {cpp_log_values['p_map_frame_local'][2]:.3f}")
    print("-" * 30)


    # --- Visualize (using K3D) ---
    print("Preparing K3D visualization...")
    plot = k3d.plot(name=f"Frame {historical_map_seq_id} Sensor Coords", grid_visible=True)

    # Historical map points (Frame 4 in Sensor 4 coords) - make them float32 for k3d
    points_sens4_f32 = points_sens4.astype(np.float32)
    plot += k3d.points(positions=points_sens4_f32,
                       point_size=0.05, # Adjust size as needed
                       color=0xaaaaaa, # Grey
                       name=f'Historical PC (Frame {historical_map_seq_id})')

    # Projected point location (calculated in Python) - make it float32
    p_map_frame_local_py_f32 = p_map_frame_local_py.astype(np.float32)
    plot += k3d.points(positions=p_map_frame_local_py_f32.reshape(1, 3), # Reshape to (1, 3)
                       point_size=0.5, # Make it larger
                       color=0xff0000, # Red
                       name=f'Python Projection (Pt {point_original_index} from Fr {current_frame_seq_id})')

    # Projected point location (from C++ log) - make it float32
    p_map_frame_local_cpp_f32 = np.array(cpp_log_values['p_map_frame_local'], dtype=np.float32)
    plot += k3d.points(positions=p_map_frame_local_cpp_f32.reshape(1, 3), # Reshape to (1, 3)
                       point_size=0.5, # Make it larger
                       color=0x00ff00, # Green
                       name=f'C++ Projection (Pt {point_original_index} from Fr {current_frame_seq_id})')

    # Origin marker (Coordinate Frame Axes) for Sensor 4 frame
    axis_length = 2.0 # Length of axes lines
    origin = np.array([0, 0, 0], dtype=np.float32)
    x_axis_end = np.array([axis_length, 0, 0], dtype=np.float32)
    y_axis_end = np.array([0, axis_length, 0], dtype=np.float32)
    z_axis_end = np.array([0, 0, axis_length], dtype=np.float32)

    plot += k3d.line([origin, x_axis_end], color=0xff0000, width=0.02, name='X Axis') # Red X
    plot += k3d.line([origin, y_axis_end], color=0x00ff00, width=0.02, name='Y Axis') # Green Y
    plot += k3d.line([origin, z_axis_end], color=0x0000ff, width=0.02, name='Z Axis') # Blue Z

    plot.display() # Display inline if no filename specified

    print("-" * 30)
    print("--- Python Verification Complete ---")

if __name__ == "__main__":
    main()

--- Starting Python Verification ---
NuScenes Root: /datasets/nuscenes, Version: v1.0-mini
Scene Index: 0
Current Frame Seq ID: 5
Historical Map Seq ID: 4
Point Original Index: 5
Using C++ Log Values:
  p_world_glob: (408.084, 1159.242, -0.086)
  map_project_T: (15.703, 1231.094, 43.124)
  p_map_frame_local: (-4.148, 3.102, -1.75)
------------------------------
Loading NuScenes (v1.0-mini from /datasets/nuscenes)...
Getting sample tokens...
Current Lidar Token (Frame 5): fca0353274424116948fa1f10be1fc29
Historical Lidar Token (Frame 4): 6a97481174074729a9d0ffa096eaa498
------------------------------
Calculating transformations...
Loading point clouds...
Extracting point 5 from Frame 5...
------------------------------
Verification 1: Global Point Coordinates
  Python calculated p_world_glob: (408.084, 1159.242, -0.086)
  C++ logged p_world.glob:      (408.084, 1159.242, -0.086)
  Difference (Py - C++):        (-0.000, -0.000, -0.000)
  >>> Global coordinates MATCH.
--------------------

Output()

------------------------------
--- Python Verification Complete ---
